In [20]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import seaborn as sns
from sodapy import Socrata
from datetime import datetime
import os
from tqdm import tqdm
import sys
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# sys.path.append("../src")
sys.path.append(str(Path.cwd().parent))
import utils

path = os.getcwd()
print(path)

path_repo = "C:/Users/PC/Desktop/tesis/"
path_data = "C:/Users/PC/Desktop/tesis/data/"

c:\Users\PC\Desktop\python\rates_degree_work


- La tabla es semanal, terminando en cada viernes

In [21]:
tasascap = pd.read_csv(path_repo + "/data/2022.csv", decimal = ",", thousands = ".", encoding='utf-8')
tasascap.columns = [x.lower().replace(" ","_") for x in tasascap.columns]
tasascap.rename(columns = {"tasa_efectiva_promedio_ponderada":"tasa_ponderada", "montos_desembolsados":"monto", "numero_de_creditos_desembolsados":"num_creditos", 
                           "sum_numero_de_creditos":"num_creditos", "producto_de_crédito":"producto", "plazo_de_crédito":"plazo", "nombre_entidad":"banco"}, inplace=True)
tasascap.drop(columns = ["tipo_entidad", "codigo_entidad"], inplace =True)
tasascap.sample(5)

,nombre_tipo_entidad,banco,fecha_corte,tipo_de_persona,sexo,tamaño_de_empresa,tipo_de_crédito,tipo_de_garantía,producto,plazo,tasa_ponderada,margen_adicional,monto,num_creditos,grupo_etnico,antiguedad_de_la_empresa,tipo_de_tasa,rango_monto_desembolsado,clase_deudor,codigo_ciiu,codigo_municipio
98072,BC-ESTABLECIMIENTO BANCARIO,Bancolombia,16/12/2022,Jurídica,No aplica,Mediana empresa,Comercial ordinario,Garantia idónea o no idónea,Empresarial en pesos,Más de 3 años y hasta 5 años,13.2900,0.0000,2020690716.0000,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31581,BC-ESTABLECIMIENTO BANCARIO,Bancolombia,26/08/2022,Natural,Femenino,No aplica,Consumo,Garantia idónea o no idónea,Consumo empleados,Más de 3 años y hasta 5 años,10.4400,0.0000,1641407768.0000,62,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83851,BC-ESTABLECIMIENTO BANCARIO,Bancolombia,25/11/2022,Natural,Masculino,No aplica,Consumo,Sin garantia,Créditos de consumo de bajo monto,Hasta 6 meses,43.2400,0.0000,214835000.0000,207,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54347,BC-ESTABLECIMIENTO BANCARIO,Coopcentral,30/09/2022,Jurídica,No aplica,Mediana empresa,Comercial ordinario,Sin garantia,Empresarial en pesos,Más de 3 años y hasta 5 años,16.0700,0.0000,4687700000.0000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84172,BC-ESTABLECIMIENTO BANCARIO,Banco Popular,25/11/2022,Natural,Femenino,No aplica,Consumo,Sin garantia,Tarjeta de crédito para ingresos hasta 2 SMMLV,Avances en efectivo,30.2600,0.0000,128605973.0000,136,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
cols_cat = ['nombre_tipo_entidad', 'banco', 'fecha_corte', 'tipo_de_persona', 'sexo', 'tamaño_de_empresa', 'tipo_de_crédito', 'tipo_de_garantía',
            'producto', 'plazo', 'grupo_etnico', 'antiguedad_de_la_empresa', 'tipo_de_tasa', 'clase_deudor', 'codigo_ciiu', 'codigo_municipio']
cols_cont = ['tasa_ponderada', 'margen_adicional', 'monto', 'num_obl', 'rango_monto_desembolsado']

tasascap["fecha_corte"] = pd.to_datetime(tasascap["fecha_corte"], dayfirst=True, errors='coerce')

tasascap["tasa_ponderada"] = tasascap["tasa_ponderada"].astype(float)
tasascap["num_creditos"] = tasascap["num_creditos"].astype(int)
tasascap["fecha_corte"] = pd.to_datetime(tasascap["fecha_corte"])
tasascap["anomes"] = tasascap["fecha_corte"].dt.strftime("%Y-%m")

# tasascap = tasascap[~tasascap["banco"].isin(["Fondo Nacional del Ahorro", "La Hipotecaria", "Bancoldex", "Confiar",
#                                         "IRIS C.F.", "Bancamía S.A.", "Bancien", "Mibanco S.A.", "Banco Contactar",
#                                         'Tuya','Crezcamos','Credifamilia','Financiera Juriscoop C.F.','Rappipay',
#                                         'Banco GNB Sudameris','Banco Popular','Coopcentral','Banco Pichincha S.A.','Nu Colombia C.F.',
#                                         "Caja de Vivienda Militar y de Polícia"])] # Excluidos por tasas atipicos
                                        
# tasascap = tasascap[tasascap["tasa_ponderada"] <= 50]
# tasascap = tasascap[tasascap["num_creditos"] >= 2]
# tasascap = tasascap[(tasascap["rango_monto_desembolsado"].notna())] # tienen nulos antes de 202309

tasascap ["banco"] = tasascap["banco"].str.replace("Compañía De Financiamiento","CF").str.replace("Cooperativa Financiera","CF")
tasascap["plazo"] = tasascap["plazo"].replace({'Más de 1 año y hasta 3 años':"(1-3]", 'Más de 3 años y hasta 5 años':"(3-5]",
                                                'Más de 5 años y hasta 7 años':"(5-7]",
                                                'Más de 7 años y hasta 10 años':"(7-10]", 'Más de 10 años':"(10-]",
                                                "Hasta 6 meses":"(0-1]", "Más de 6 meses y hasta 1 año":"(0-1]",
                                                'Hasta 30 días':"(0-1]", 'Entre 31 días y 1 año':"(0-1]"})

tasascap["prom_monto_porobl"] = tasascap["monto"]/tasascap["num_creditos"]

print(tasascap["fecha_corte"].min(), "|", tasascap["fecha_corte"].max())

print(tasascap.shape)
print("Bancos:", tasascap["banco"].nunique())

tasascap.sample(7)

2022-07-01 00:00:00 | 2022-12-30 00:00:00
(107151, 23)
Bancos: 49


,nombre_tipo_entidad,banco,fecha_corte,tipo_de_persona,sexo,tamaño_de_empresa,tipo_de_crédito,tipo_de_garantía,producto,plazo,tasa_ponderada,margen_adicional,monto,num_creditos,grupo_etnico,antiguedad_de_la_empresa,tipo_de_tasa,rango_monto_desembolsado,clase_deudor,codigo_ciiu,codigo_municipio,anomes,prom_monto_porobl
92603,BC-ESTABLECIMIENTO BANCARIO,Banco Popular,2022-12-09,Natural,Femenino,Pequeña empresa,Comercial ordinario,Sin garantia,Tarjeta de crédito empresarial,Consumos entre 2 y 6 meses,41.4000,0.0000,3795915.0000,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-12,542273.5714
58004,BC-ESTABLECIMIENTO BANCARIO,Banco Caja Social S.A.,2022-10-07,Natural,Masculino,Microempresa,Comercial ordinario,Sin garantia,Tarjeta de crédito empresarial,Consumos entre 2 y 6 meses,32.0900,0.0000,7656902.0000,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-10,1914225.5000
46432,BC-ESTABLECIMIENTO BANCARIO,Bancoomeva,2022-09-16,Jurídica,No aplica,Microempresa,Comercial ordinario,Sin garantia,Tarjeta de crédito empresarial,Avances en efectivo,35.1300,0.0000,1100000.0000,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-09,275000.0000
54592,BC-ESTABLECIMIENTO BANCARIO,BBVA Colombia,2022-09-30,Jurídica,No aplica,Mediana empresa,Comercial ordinario,Garantia idónea o no idónea,Empresarial en pesos,(1-3],16.8000,0.0000,19790786882.0000,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-09,1979078688.2000
15827,BC-ESTABLECIMIENTO BANCARIO,Finandina,2022-07-29,Natural,Femenino,No aplica,Consumo,Sin garantia,Libre inversión,(3-5],25.9700,0.0000,2843300000.0000,168,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-07,16924404.7619
81731,BC-ESTABLECIMIENTO BANCARIO,Banco Caja Social S.A.,2022-11-18,Natural,Femenino,Microempresa,Microcredito,Sin garantia,Microcrédito menor a 25 SMMLV,(3-5],38.5700,0.0000,304401000.0000,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-11,15220050.0000
53487,CF-COMPAÑÍA DE FINANCIAMIENTO,Coltefinanciera,2022-09-30,Natural,Femenino,Microempresa,Comercial ordinario,Sin garantia,Otros Comercial,(3-5],33.1800,0.0000,244170000.0000,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-09,40695000.0000


In [23]:
tasascap[tasascap["anomes"] == "2022-08"]["fecha_corte"].value_counts()

fecha_corte
2022-08-26    4077
2022-08-05    3956
2022-08-12    3906
2022-08-19    3833
Name: count, dtype: int64

In [24]:
utils.vc(tasascap["tipo_de_crédito"])

,tipo_de_crédito,count,porc
0,Consumo,46568,43.4610
1,Comercial ordinario,38828,36.2374
2,Microcredito,9314,8.6926
3,Vivienda,8160,7.6156
4,Comercial preferencial o corporativo,3197,2.9837
5,Comercial tesoreria,646,0.6029
6,Comercial especial,436,0.4069


In [25]:
utils.vc(tasascap["tipo_de_garantía"])

,tipo_de_garantía,count,porc
0,Sin garantia,55212,51.5282
1,Garantia idónea o no idónea,43931,40.9999
2,Garantía fondo nacional de garantías (FNG) o Fondo de Garantías de Antioquia (FGA),5582,5.2096
3,Garantía del fondo agropecuario de garantías (FAG),2424,2.2623


In [26]:
tasascap.isnull().mean()

nombre_tipo_entidad        0.0000
banco                      0.0000
fecha_corte                0.0000
tipo_de_persona            0.0000
sexo                       0.0000
tamaño_de_empresa          0.0000
tipo_de_crédito            0.0000
tipo_de_garantía           0.0000
producto                   0.0000
plazo                      0.0000
tasa_ponderada             0.0000
margen_adicional           0.0000
monto                      0.0000
num_creditos               0.0000
grupo_etnico               1.0000
antiguedad_de_la_empresa   1.0000
tipo_de_tasa               1.0000
rango_monto_desembolsado   1.0000
clase_deudor               1.0000
codigo_ciiu                1.0000
codigo_municipio           1.0000
anomes                     0.0000
prom_monto_porobl          0.0000
dtype: float64

In [27]:
# Sample
tmp = tasascap.drop(columns = ['margen_adicional', 'grupo_etnico', 'antiguedad_de_la_empresa', 'rango_monto_desembolsado', 'clase_deudor', 'codigo_ciiu',
       'codigo_municipio', 'anomes', 'prom_monto_porobl']).sample(100)
tmp.sample(5)

,nombre_tipo_entidad,banco,fecha_corte,tipo_de_persona,sexo,tamaño_de_empresa,tipo_de_crédito,tipo_de_garantía,producto,plazo,tasa_ponderada,monto,num_creditos,tipo_de_tasa
40899,BC-ESTABLECIMIENTO BANCARIO,Bancamía S.A.,2022-09-09,Natural,Femenino,Pequeña empresa,Comercial ordinario,Sin garantia,Empresarial en pesos,(3-5],35.2500,40172550.0000,1,NaN
25039,BC-ESTABLECIMIENTO BANCARIO,Banagrario,2022-08-12,Natural,Masculino,Microempresa,Comercial ordinario,Garantía fondo nacional de garantías (FNG) o Fondo de Garantías de Antioquia (FGA),Otros Comercial,(1-3],20.6000,325000000.0000,7,NaN
21051,BC-ESTABLECIMIENTO BANCARIO,Itaú,2022-08-05,Natural,Femenino,No aplica,Consumo,Garantia idónea o no idónea,Sobregiro en cuenta corriente persona natural,(0-1],32.9000,909890956.5300,1487,NaN
98872,BC-ESTABLECIMIENTO BANCARIO,Mibanco S.A.,2022-12-23,Natural,Femenino,Microempresa,Microcredito,Sin garantia,Microcrédito entre 25 y 120 SMMLV,(1-3],40.5800,2098426092.0000,56,NaN
89824,BC-ESTABLECIMIENTO BANCARIO,Bancolombia,2022-12-02,Jurídica,No aplica,Pequeña empresa,Comercial ordinario,Garantia idónea o no idónea,Leasing financiero - otros,(5-7],21.6200,516600000.0000,3,NaN


In [29]:
print(np.percentile(tasascap["tasa_ponderada"], 2.25))
print(np.percentile(tasascap["tasa_ponderada"], 5))
print(np.percentile(tasascap["tasa_ponderada"], 50))
print(np.percentile(tasascap["tasa_ponderada"], 95))
print(np.percentile(tasascap["tasa_ponderada"], 97.5))

4.91
10.7
22.66
41.125
44.25


In [ ]:
print(tasascap[tasascap["tasa_ponderada"] <= 10].shape)
print(tasascap[tasascap["tasa_ponderada"] >= 40].shape)

In [ ]:
# ax = tasascap[tasascap["fecha_corte"] >= "2025-01-01"]
ax = tasascap
ax = ax.groupby("producto").agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax.columns = ["producto", "monto", "num_creditos", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_creditos"]/ax["num_creditos"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_creditos"]
ax = ax.sort_values("num_creditos", ascending=False)#.head(30)
ax

In [ ]:
# ax = tasascap[tasascap["fecha_corte"] >= "2025-01-01"]
ax = tasascap
ax = ax.groupby("banco").agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax.columns = ["banco", "monto", "num_creditos", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_creditos"]/ax["num_creditos"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_creditos"]
ax = ax.sort_values("num_creditos", ascending=False)#.head(30)
ax

In [ ]:
tasascap["anomes"].value_counts(dropna = False, normalize =True)
ax = tasascap
ax = ax.groupby(["anomes"]).agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax.columns = ["mes", "monto", "num_creditos", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_creditos"]/ax["num_creditos"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_creditos"]
#ax.sort_values("monto", ascending=False).head(40)
ax

In [ ]:
ax = tasascap
ax = ax.groupby(["anomes", "plazo", "rango_desembolso"]).agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":"mean"}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax["monto"] = ax["monto"]/1000000
ax["prom_monto_porobl"] = ax["monto"]/ax["num_creditos"] # millones

#ax.sort_values("monto", ascending=False).head(40)
tasascap = ax
tasascap

## How to group?

## BANREP

In [12]:
banrep1 = pd.read_excel(path_data+"BANREP/tasas de interes y sector - tasas interes.xlsx", decimal = ",")
banrep1 = banrep1.iloc[1:]
banrep1.columns = [x.lower().replace("í","i").replace("ó","o").replace("é","e").replace(" ", "_") for x in banrep1.columns]
banrep1 = banrep1.replace({'-':np.nan})
banrep1["fecha"] = pd.to_datetime(banrep1["fecha"], format="%d/%m/%Y", errors='coerce')
banrep1["periodo"] = banrep1["fecha"].dt.strftime("%Y-%m")

print(banrep1["fecha"].min(), "|", banrep1["fecha"].max())
print(banrep1.shape)
banrep1

2022-01-03 00:00:00 | 2026-10-16 00:00:00
(1196, 22)


,fecha,"credito_de_consumo,_tasa_de_interes,_semanal(dato_semanal)","microcreditos_(diferentes_de_leasing),_tasa_de_interes,_semanal(dato_semanal)","creditos_comerciales_(ordinario),_tasa_de_interes,_semanal(dato_semanal)","creditos_comerciales_(preferencial_o_corporativo),_tasa_de_interes,_semanal(dato_semanal)","indicador_bancario_de_referencia_(ibr)_overnight,_efectiva(dato_diario)","indicador_bancario_de_referencia_(ibr)_overnight,_efectiva(promedio_semanal)","indicador_bancario_de_referencia_(ibr)_a_1_mes,_efectiva(dato_diario)","indicador_bancario_de_referencia_(ibr)_a_1_mes,_efectiva(promedio_semanal)","indicador_bancario_de_referencia_(ibr)_a_3_meses,_efectiva(dato_diario)","indicador_bancario_de_referencia_(ibr)_a_3_meses,_efectiva(promedio_semanal)",operaciones_interbancarias_-_monto(dato_diario),"indicador_bancario_de_referencia_(ibr)_a_6_meses,_efectiva(dato_diario)","indicador_bancario_de_referencia_(ibr)_a_6_meses,_efectiva(promedio_semanal)","indicador_bancario_de_referencia_(ibr)_a_12_meses,_efectiva(dato_diario)","indicador_bancario_de_referencia_(ibr)_a_12_meses,_efectiva(promedio_semanal)","credito_productivo_(total),_semanal(dato_semanal)","credito_productivo_de_mayor_monto(total),_semanal(dato_semanal)","credito_popular_productivo_(total),_mensual(promedio_mensual)","captaciones_a_traves_de_cdt_por_red_de_oficinas,_diaria(dato_diario)","captaciones_a_traves_de_cdt_por_tesoreria,_diaria(dato_diario)",periodo
1,2022-01-03,NaN,NaN,NaN,NaN,2.998,NaN,3.087,NaN,3.586,NaN,"915.900,00",4.274,NaN,NaN,NaN,NaN,NaN,NaN,3.49,4.34,2022-01
2,2022-01-04,NaN,NaN,NaN,NaN,2.981,NaN,3.140,NaN,3.644,NaN,"715.500,00",4.335,NaN,NaN,NaN,NaN,NaN,NaN,3.73,5.23,2022-01
3,2022-01-05,NaN,NaN,NaN,NaN,2.995,NaN,3.168,NaN,3.660,NaN,"745.600,00",4.377,NaN,NaN,NaN,NaN,NaN,NaN,3.75,4.02,2022-01
4,2022-01-06,NaN,NaN,NaN,NaN,2.998,NaN,3.218,NaN,3.711,NaN,"835.300,00",4.512,NaN,NaN,NaN,NaN,NaN,NaN,3.53,5.13,2022-01
5,2022-01-07,16.87,39.24,9.17,6.20,3.000,2.994,3.286,3.180,3.773,3.675,"979.300,00",4.611,4.422,NaN,NaN,NaN,NaN,NaN,3.73,4.52,2022-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1192,2026-10-15,NaN,NaN,NaN,NaN,12.260,NaN,12.332,NaN,12.388,NaN,"1.207.853,08",12.481,NaN,12.599,NaN,NaN,NaN,NaN,10.70,12.92,2026-10
1193,2026-10-16,NaN,NaN,NaN,NaN,12.257,12.257,12.295,12.295,12.382,12.382,NaN,12.468,12.468,12.563,12.563,NaN,NaN,NaN,NaN,NaN,2026-10
1194,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1195,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
